In [ ]:
import pandas as pd

# Load the dataset
try:
    df = pd.read_csv('test_data.csv')

    # Select the 7 attributes
    cols = [
        'Ride_Distance(km)', 'Ride_Time(min)', 'Customer_Rating',
        'Driver_Rating', 'Vehicle_Type_Bike', 'Payment_Method_UPI', 'Booking_Value'
    ]
    df = df[cols].copy()

    print("Dataset loaded successfully!")
    print("First few rows of the dataset:")
    print(df.head())
except FileNotFoundError:
    print("File not found. Please check the file path or upload test_data.csv.")
except Exception as e:
    print(f"An error occurred: {e}")

Dataset loaded successfully!
First few rows of the dataset:
   Ride_Distance(km)  Ride_Time(min)  Customer_Rating  Driver_Rating  \
0               6.58              11              3.6            4.2   
1              16.35              14              4.2            4.0   
2               1.90              23              3.8            3.3   
3               3.48              49              3.7            3.3   
4              14.15              35              3.8            3.5   

   Vehicle_Type_Bike  Payment_Method_UPI  Booking_Value  
0               True               False         138.98  
1               True               False         276.89  
2              False                True         126.07  
3              False               False         201.75  
4               True                True         276.75  


In [ ]:
from mlxtend.preprocessing import TransactionEncoder

# Discretize the 5 continuous numeric columns into 3 equal-width bins
num_cols = ['Ride_Distance(km)', 'Ride_Time(min)', 'Customer_Rating', 'Driver_Rating', 'Booking_Value']
df_disc = df.copy()
for col in num_cols:
    df_disc[col] = pd.cut(df_disc[col], bins=3).astype(str)

# Convert each ride record into a list of "Attribute=Bin" items
transaction_list = [
    [f"{col}={row[col]}" for col in df_disc.columns]
    for _, row in df_disc.iterrows()
]

# Convert transaction list to boolean matrix using TransactionEncoder
te = TransactionEncoder()
te_ary = te.fit(transaction_list).transform(transaction_list)

# Convert to DataFrame with item names as columns
encoded_df = pd.DataFrame(te_ary, columns=te.columns_)

# Display the first few rows of the encoded DataFrame
print("\nFirst few rows of the encoded DataFrame:")
print(encoded_df.head())
print(f"\nTotal number of transactions: {len(encoded_df)}")
print(f"Number of unique items: {len(encoded_df.columns)}")


First few rows of the encoded DataFrame:
   Booking_Value=(190.843, 333.077]  Booking_Value=(333.077, 475.31]  \
0                             False                            False   
1                              True                            False   
2                             False                            False   
3                              True                            False   
4                              True                            False   

   Booking_Value=(48.183, 190.843]  Customer_Rating=(2.998, 3.667]  \
0                             True                            True   
1                            False                           False   
2                             True                           False   
3                            False                           False   
4                            False                           False   

   Customer_Rating=(3.667, 4.333]  Customer_Rating=(4.333, 5.0]  \
0                           False    

In [ ]:
from mlxtend.frequent_patterns import apriori, association_rules

# Apply Apriori to find frequent itemsets (min_support = 0.20, matching Weka)
frequent_itemsets = apriori(encoded_df, min_support=0.20, use_colnames=True)

# Generate association rules (min_threshold = 0.70 confidence, matching Weka)
rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.70)

# Filter rules with lift >= 1 and sort by confidence and lift
rules = rules[rules['lift'] >= 1.0].sort_values(by=['confidence', 'lift'], ascending=False)

# Display summary counts and top 10 association rules
print(f"Number of frequent itemsets found: {len(frequent_itemsets)}")
print(f"Number of rules generated: {len(rules)}")
print("\nTop 10 association rules (sorted by confidence & lift):")
print(rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(10).to_string(index=False))

Number of frequent itemsets found: 40
Number of rules generated: 15

Top 10 association rules (sorted by confidence & lift):
                                                          antecedents                        consequents  support  confidence     lift
                                    (Booking_Value=(333.077, 475.31]) (Ride_Distance(km)=(17.167, 25.0]) 0.209667    0.914909 2.823793
                                    (Booking_Value=(48.183, 190.843]) (Ride_Distance(km)=(1.476, 9.333]) 0.241333    0.911265 2.661923
          (Ride_Distance(km)=(9.333, 17.167], Vehicle_Type_Bike=True) (Booking_Value=(190.843, 333.077]) 0.208833    0.875611 1.730457
                                  (Ride_Distance(km)=(9.333, 17.167]) (Booking_Value=(190.843, 333.077]) 0.290667    0.871129 1.721599
(Ride_Distance(km)=(9.333, 17.167], Booking_Value=(190.843, 333.077])           (Vehicle_Type_Bike=True) 0.208833    0.718463 1.019820
                                       (Ride_Time(min)=(4.946, 23

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
